# Object tracking in self-recorded videos (CoTracker3)

Pick one or more points on an object in one frame; the notebook tracks them through the whole video,
draws the trajectories, saves the annotated video and a CSV of positions, and prints metrics.

All the logic lives in `src/tracker/`; this notebook only sets parameters and shows results.
The first run downloads the CoTracker3 model (~100 MB) via `torch.hub`.

In [ ]:
import os, sys

# Make `src/` importable when running from the notebooks/ folder (not needed after `pip install -e .`)
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))

import matplotlib.pyplot as plt
import numpy as np
from tracker import load_video, make_queries, track_video, format_metrics

## 1. Settings
`POINTS` are `(x, y)` pixel coordinates in the **original** video, selected on `QUERY_FRAME`.
Use the grid in the next cell to find good coordinates.

In [ ]:
VIDEO = os.path.join(ROOT, "examples", "bicycle.mp4")
QUERY_FRAME = 70
POINTS = [
    (125, 50),
    # (150, 80),   # add more points on the same object for a more robust track
]

MAX_SIDE = 640          # downscale frames larger than this (None = keep original size)
OUTPUT_VIDEO = os.path.join(ROOT, "results", "bicycle_tracked.mp4")
OUTPUT_CSV = os.path.join(ROOT, "results", "bicycle_tracks.csv")

## 2. Check the selected points

In [ ]:
video = load_video(VIDEO, max_side=MAX_SIDE)
queries = make_queries(POINTS, QUERY_FRAME, video.scale, video.num_frames)
print(f"{video.num_frames} frames, {video.fps:.2f} FPS, original {video.original_size}, loaded {video.size}")

def show_points(frame, queries, step=50, title=""):
    plt.figure(figsize=(10, 6))
    plt.imshow(frame)
    plt.scatter(queries[:, 1], queries[:, 2], c="cyan", s=30)
    for i, (_, x, y) in enumerate(queries):
        plt.text(x + 5, y - 5, str(i), color="cyan", fontsize=9, weight="bold")
    H, W, _ = frame.shape
    plt.xticks(np.arange(0, W + 1, step)); plt.yticks(np.arange(0, H + 1, step))
    plt.title(title); plt.grid(alpha=0.3); plt.show()

# Note: the axes show LOADED-frame pixels; divide by video.scale for original pixels.
show_points(video.frames[QUERY_FRAME], queries, title=f"Query points on frame {QUERY_FRAME}")

## 3. Track

In [ ]:
result = track_video(VIDEO, POINTS, query_frame=QUERY_FRAME, max_side=MAX_SIDE,
                     output_path=OUTPUT_VIDEO, csv_path=OUTPUT_CSV)
print("Saved video:", result.output_path)
print("Saved CSV  :", result.csv_path)

## 4. Results
Without ground truth these metrics describe how the track *behaves*, not how accurate it is.
Pass `gt_tracks=` / `gt_visibility=` to `tracker.evaluate_tracking` for MSE, EPE, PCK and visibility accuracy.

In [ ]:
name = os.path.basename(VIDEO)
print(format_metrics(result.metrics, f"{name} (smoothed)"))
print(format_metrics(result.raw_metrics, f"{name} (raw)"))

# Trajectory of each point over time (visible frames only)
vis = result.visibility > 0.5
plt.figure(figsize=(10, 4))
for p in range(result.smoothed.shape[1]):
    t = np.where(vis[:, p])[0]
    plt.plot(t, result.smoothed[t, p, 0] / video.scale, ".", ms=2, label=f"point {p} x")
    plt.plot(t, result.smoothed[t, p, 1] / video.scale, ".", ms=2, label=f"point {p} y")
plt.xlabel("frame"); plt.ylabel("pixels (original video)"); plt.legend(); plt.title("Position over time"); plt.show()